# Make Matrix ID Table

In [ ]:
from __future__ import annotations

import csv
import os

import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
main_dir = os.path.dirname(os.getcwd())
title_info_tsv_path = os.path.join(main_dir, "data/output/wiki_title_info.tsv")
summary_tsv_path = os.path.join(main_dir, "data/output/wiki_summary.tsv")
matrix_index_info_tsv_path = os.path.join(
    main_dir, "data/output/wiki_matrix_index_info.tsv"
)

In [ ]:
title_info_df = pd.read_csv(
    title_info_tsv_path, sep="\t", header=0, quoting=csv.QUOTE_NONE
)

title_info_df.head(20)

In [ ]:
article_count = (~title_info_df["IsRedirect"]).sum()

In [ ]:
page_id_ary = np.full(article_count, -1, dtype=int)
matrix_id_ary = np.full(article_count, -2, dtype=int)
page_title_ary = [""] * article_count
offset_ary = np.full(article_count, -1, dtype=int)

In [ ]:
offset = 0

with open(summary_tsv_path, "rb") as sum_f:
    offset += len(sum_f.readline())  # Skip header
    for n, line in enumerate(tqdm(sum_f, total=article_count, desc="Reading Summary")):
        offset_ary[n] = offset
        offset += len(line)
        list_line = line.decode("utf-8").split("\t")
        page_id_ary[n] = int(list_line[0])
        matrix_id_ary[n] = int(list_line[1])
        page_title_ary[n] = list_line[2]

In [ ]:
assert title_info_df["MatrixIndex"].max() == matrix_id_ary.max()
assert np.all(np.diff(matrix_id_ary) == 1)
assert matrix_id_ary[0] == 1

In [ ]:
matrix_index_info_df = pd.DataFrame(
    {
        "PageID": page_id_ary,
        "MatrixIndex": matrix_id_ary,
        "PageTitle": page_title_ary,
        "Offset": offset_ary,
    }
)

matrix_index_info_df.head(20)

In [ ]:
matrix_index_info_df.to_csv(
    matrix_index_info_tsv_path, sep="\t", header=True, index=False
)